# 03.2 — Logistic Regression for Text

**Problem it solves:** Text classification with a discriminative model — directly learns the decision boundary P(y|x) rather than modeling the data distribution.

**Why it beats NB:** NB assumes feature independence. LR relaxes this — it can learn that 'not good' is negative even though 'good' alone is positive. Given enough data, LR almost always beats NB.

---

In [ ]:
import numpy as np
import re
from collections import Counter

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class BagOfWordsVectorizer:
    """Convert texts to binary or count bag-of-words vectors."""
    def __init__(self, binary=False, min_df=1):
        self.binary = binary
        self.min_df = min_df
        self.vocab = {}
    
    def fit(self, texts):
        df = Counter()
        for text in texts:
            df.update(set(tokenize(text)))
        valid = [w for w, f in df.items() if f >= self.min_df]
        self.vocab = {w: i for i, w in enumerate(sorted(valid))}
        return self
    
    def transform(self, texts):
        X = np.zeros((len(texts), len(self.vocab)))
        for i, text in enumerate(texts):
            counts = Counter(tokenize(text))
            for word, count in counts.items():
                if word in self.vocab:
                    X[i, self.vocab[word]] = 1 if self.binary else count
        return X

class LogisticRegression:
    """Binary logistic regression trained with gradient descent."""
    
    def __init__(self, lr=0.1, n_epochs=100, lambda_l2=0.01):
        self.lr = lr
        self.n_epochs = n_epochs
        self.lambda_l2 = lambda_l2
        self.weights = None
        self.bias = 0.0
        self.losses = []
    
    def fit(self, X: np.ndarray, y: np.ndarray):
        n, d = X.shape
        self.weights = np.zeros(d)
        self.bias = 0.0
        
        for epoch in range(self.n_epochs):
            # Forward pass
            z = X @ self.weights + self.bias
            y_hat = sigmoid(z)
            
            # Binary cross-entropy loss + L2 regularization
            eps = 1e-10
            bce = -np.mean(y * np.log(y_hat + eps) + (1-y) * np.log(1-y_hat + eps))
            l2 = self.lambda_l2 * np.sum(self.weights**2)
            loss = bce + l2
            self.losses.append(loss)
            
            # Gradients
            error = y_hat - y          # (n,)
            dw = (X.T @ error) / n + 2 * self.lambda_l2 * self.weights
            db = np.mean(error)
            
            # Update
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db
            
            if epoch % 20 == 0:
                preds = (y_hat >= 0.5).astype(int)
                acc = np.mean(preds == y)
                print(f"  epoch {epoch:3d}: loss={loss:.4f}  train_acc={acc:.3f}")
        
        return self
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return sigmoid(X @ self.weights + self.bias)
    
    def predict(self, X: np.ndarray, threshold=0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def top_features(self, vocab: dict, n=10):
        """Most positive and negative features."""
        idx2word = {v:k for k,v in vocab.items()}
        pos = sorted(range(len(self.weights)), key=lambda i: self.weights[i], reverse=True)
        neg = sorted(range(len(self.weights)), key=lambda i: self.weights[i])
        print("Most positive features:")
        for i in pos[:n]:
            print(f"  {idx2word[i]:20} w={self.weights[i]:+.4f}")
        print("Most negative features:")
        for i in neg[:n]:
            print(f"  {idx2word[i]:20} w={self.weights[i]:+.4f}")

# Dataset
train_texts = [
    "great movie loved it wonderful",
    "fantastic film amazing performances",
    "boring and terrible waste of time",
    "awful film hated every moment",
    "not good at all very disappointing",
    "surprisingly good film enjoyed it",
    "best movie of the year must watch",
    "dreadful acting poor script avoid it",
    "heartwarming and inspiring beautiful story",
    "horrible movie completely unwatchable",
]
train_labels = [1,1,0,0,0,1,1,0,1,0]  # 1=positive, 0=negative

vec = BagOfWordsVectorizer().fit(train_texts)
X_train = vec.transform(train_texts)
y_train = np.array(train_labels, dtype=float)

print(f"Feature matrix: {X_train.shape}")
model = LogisticRegression(lr=0.5, n_epochs=100, lambda_l2=0.01)
model.fit(X_train, y_train)

In [ ]:
# Inspect learned weights
model.top_features(vec.vocab, n=8)

# Test predictions
test_texts = [
    "wonderful performances loved the story",
    "terrible boring awful film",
    "not bad actually quite good",  # NB would miss 'not'
    "not good at all horrible",     # explicit negation
]
X_test = vec.transform(test_texts)
probs = model.predict_proba(X_test)

print("\nTest predictions:")
for text, prob in zip(test_texts, probs):
    label = 'positive' if prob >= 0.5 else 'negative'
    print(f"  {prob:.3f} ({label:8}) {text!r}")

In [ ]:
# Bigrams: capturing 'not good' as a single feature
def tokenize_ngrams(text, n=2):
    tokens = re.findall(r'\b[a-z]+\b', text.lower())
    unigrams = tokens
    bigrams = [f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens)-1)]
    return unigrams + bigrams

class NgramVectorizer(BagOfWordsVectorizer):
    def __init__(self, n=2, **kwargs):
        super().__init__(**kwargs)
        self.n = n

# Show bigrams found in the training data
all_bigrams = []
for text in train_texts:
    tokens = re.findall(r'\b[a-z]+\b', text.lower())
    all_bigrams.extend([f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens)-1)])

print("Sample bigrams that capture sentiment context:")
for bg in all_bigrams:
    if any(neg in bg for neg in ['not_', '_not', 'awful', 'great', 'loved', 'terrible']):
        print(f"  {bg}")

## Summary

**LR vs NB:**
- LR is a discriminative model: learns P(y|x)
- NB is a generative model: learns P(x|y) then uses Bayes
- LR needs more data to converge but achieves higher accuracy
- NB is faster to train and requires fewer samples

**What breaks LR:**
- High-dimensional sparse features → needs regularization
- Long-range dependencies (e.g., negation across a sentence) — need n-grams or neural models
- Class imbalance → adjust threshold or use class weights
- Collinear features → L2 regularization helps

---
**Next:** 03.3 — N-gram Language Models